https://hsb-rhein-waal.digibib.net/search/katalog/record/(DE-605)99371417100806441?be-katalog-fq=dc.type%3AAmtliche+Druckschrift&be-katalog-fq=introx.subject%3ATrade+Policy&be-katalog-sort=relevance&q-ky=%22International+trade%22&start=-19&count=20&hitcount=62&pos=5 

In [2]:
import pandas as pd 

df = pd.read_csv('../../data/raw/v2_raw_imf_trade.csv')

In [3]:
df['COUNTERPART_COUNTRY'].str.contains('United States').sum()

np.int64(6275)

In [4]:
# Correct way with proper parentheses
count = len(df[(df['COUNTERPART_COUNTRY'] == 'United States') & (df['TIME_PERIOD'] == 2005)])
print(f"Number of instances: {count}")
random_us_2005 = df[(df['COUNTERPART_COUNTRY'] == 'United States') & (df['TIME_PERIOD'] == 2005)].sample(n=5)
random_us_2005

Number of instances: 192


,COUNTRY,INDICATOR,COUNTERPART_COUNTRY,FREQUENCY,TIME_PERIOD,OBS_VALUE,SCALE
203257,Australia,"Exports of goods, Free on board (FOB), US dollar",United States,Annual,2005.0,7052.630693,Millions
346112,Barbados,"Exports of goods, Free on board (FOB), US dollar",United States,Annual,2005.0,48.378426,Millions
67702,Solomon Islands,"Exports of goods, Free on board (FOB), US dollar",United States,Annual,2005.0,0.157088,Millions
423211,Luxembourg,"Exports of goods, Free on board (FOB), US dollar",United States,Annual,2005.0,419.002517,Millions
524558,Kenya,"Exports of goods, Free on board (FOB), US dollar",United States,Annual,2005.0,228.094084,Millions


In [5]:
import pandas as pd
import numpy as np

def analyze_dataframe(df):
    """
    Comprehensive DataFrame analysis showing missing values, unique values, and value counts
    """
    
    print("=" * 80)
    print("DATAFRAME ANALYSIS")
    print("=" * 80)
    
    # Section 1: Basic Info
    print("\n1. BASIC DATAFRAME INFORMATION")
    print("-" * 40)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Total cells: {df.size}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Section 2: Missing Values Analysis
    print("\n2. MISSING VALUES BY COLUMN")
    print("-" * 40)
    
    # Create a DataFrame with missing value info
    missing_info = pd.DataFrame({
        'Column': df.columns,
        'Data Type': [df[col].dtype for col in df.columns],
        'Non-Null Count': [df[col].count() for col in df.columns],
        'Missing Count': [df[col].isnull().sum() for col in df.columns],
        'Missing %': [round(df[col].isnull().sum() / len(df) * 100, 2) for col in df.columns]
    })
    
    # Sort by missing count descending
    missing_info = missing_info.sort_values('Missing Count', ascending=False)
    
    # Format the output
    for _, row in missing_info.iterrows():
        missing_bar = '█' * int(row['Missing %'] / 5)  # Visual bar for missing percentage
        print(f"{row['Column'][:30]:<30} "
              f"| Type: {str(row['Data Type']):<10} "
              f"| Missing: {int(row['Missing Count']):>6} "
              f"| {row['Missing %']:>5}% {missing_bar}")
    
    # Section 3: Unique Values Analysis (for object/category columns)
    print("\n3. UNIQUE VALUES IN CATEGORICAL COLUMNS")
    print("-" * 40)
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    
    if len(categorical_cols) > 0:
        unique_info = pd.DataFrame({
            'Column': categorical_cols,
            'Unique Values': [df[col].nunique() for col in categorical_cols],
            'Sample Values': [str(list(df[col].dropna().unique()[:5])) for col in categorical_cols]
        })
        unique_info = unique_info.sort_values('Unique Values', ascending=False)
        
        for _, row in unique_info.iterrows():
            print(f"{row['Column'][:30]:<30} "
                  f"| Unique: {row['Unique Values']:<5} "
                  f"| Samples: {row['Sample Values'][:50]}")
    else:
        print("No categorical columns found")
    
    # Section 4: Numeric Columns Statistics
    print("\n4. NUMERIC COLUMNS SUMMARY")
    print("-" * 40)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        numeric_stats = df[numeric_cols].describe().T
        numeric_stats['missing'] = df[numeric_cols].isnull().sum()
        print(numeric_stats[['count', 'missing', 'mean', 'min', 'max']])
    else:
        print("No numeric columns found")
    
    return missing_info

def show_value_counts(df, column_name, top_n=10):
    """
    Display value counts for a specific column with formatting
    """
    print("\n" + "=" * 80)
    print(f"VALUE COUNTS FOR COLUMN: '{column_name}'")
    print("=" * 80)
    
    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in DataFrame")
        return
    
    # Get value counts
    counts = df[column_name].value_counts()
    
    # Get value counts with percentages
    counts_pct = df[column_name].value_counts(normalize=True) * 100
    
    # Create a combined DataFrame
    value_counts_df = pd.DataFrame({
        'Count': counts,
        'Percentage': counts_pct,
        'Cumulative %': counts_pct.cumsum()
    })
    
    # Show top N or all if less than top_n
    display_count = min(top_n, len(counts))
    
    print(f"\nTop {display_count} values (out of {len(counts)} unique values):")
    print("-" * 60)
    
    for i, (value, count) in enumerate(counts.head(display_count).items(), 1):
        percentage = counts_pct[value]
        bar = '█' * int(percentage / 2)  # Visual bar (max 50 chars)
        
        # Truncate value if too long
        value_str = str(value)[:30]
        
        print(f"{i:2}. {value_str:<30} "
              f"| Count: {count:<8} "
              f"| {percentage:>5.1f}% {bar}")
    
    # Show missing values if any
    missing_count = df[column_name].isnull().sum()
    if missing_count > 0:
        missing_pct = (missing_count / len(df)) * 100
        print(f"\nMissing values: {missing_count} ({missing_pct:.1f}%)")
    
    return value_counts_df

In [6]:
missing_info = analyze_dataframe(df)

# Show value counts for specific columns
show_value_counts(df, 'COUNTRY', top_n=5)
show_value_counts(df, 'TIME_PERIOD', top_n=10)
show_value_counts(df, 'City', top_n=5)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 715394 rows × 7 columns
Total cells: 5007758
Memory usage: 238.20 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
TIME_PERIOD                    | Type: float64    | Missing:    308 |  0.04% 
OBS_VALUE                      | Type: float64    | Missing:    308 |  0.04% 
SCALE                          | Type: object     | Missing:      2 |   0.0% 
COUNTERPART_COUNTRY            | Type: object     | Missing:      0 |   0.0% 
INDICATOR                      | Type: object     | Missing:      0 |   0.0% 
COUNTRY                        | Type: object     | Missing:      0 |   0.0% 
FREQUENCY                      | Type: object     | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
COUNTERPART_COUNTRY            | Unique: 214   | Samples: ['Mauritania, Islamic Republic of', 'Ukraine', 'Ba
COUNTRY                  

In [7]:
cdf = df.copy()

cdf = cdf.drop(columns=['FREQUENCY', 'SCALE', 'INDICATOR'])

cdf.rename(columns={
    'COUNTRY': 'country_a',
    'COUNTERPART_COUNTRY': 'country_b',
    'TIME_PERIOD': 'year',
    'OBS_VALUE': 'export_mill_usd',
}, inplace=True)

cdf.head()

,country_a,country_b,year,export_mill_usd
0,"Uzbekistan, Republic of","Mauritania, Islamic Republic of",2024.0,0.070536
1,Holy See,Ukraine,2020.0,0.000228
2,Holy See,Ukraine,2021.0,0.001300
3,Holy See,Ukraine,2022.0,0.000881
4,Holy See,Ukraine,2023.0,0.000108


In [8]:
missing_info = analyze_dataframe(cdf)

# Show value counts for specific columns
show_value_counts(df, 'COUNTRY', top_n=5)
show_value_counts(df, 'TIME_PERIOD', top_n=10)
show_value_counts(df, 'City', top_n=5)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 715394 rows × 4 columns
Total cells: 2861576
Memory usage: 96.40 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
export_mill_usd                | Type: float64    | Missing:    308 |  0.04% 
year                           | Type: float64    | Missing:    308 |  0.04% 
country_b                      | Type: object     | Missing:      0 |   0.0% 
country_a                      | Type: object     | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
country_b                      | Unique: 214   | Samples: ['Mauritania, Islamic Republic of', 'Ukraine', 'Ba
country_a                      | Unique: 211   | Samples: ['Uzbekistan, Republic of', 'Holy See', 'Eritrea, 

4. NUMERIC COLUMNS SUMMARY
----------------------------------------
                    count  missing         mean          min           max
year   

In [9]:
unique = cdf['year'].unique()
print(unique)

[2024. 2020. 2021. 2022. 2023. 2008. 2009. 2012. 2013. 2014. 2017. 2018.
 1997. 1998. 1999. 1992. 1993. 1994. 1995. 1996. 2000. 2002. 2003. 2006.
 2016. 2019. 2001. 2004. 2005. 2007. 2010. 2011. 2015.   nan]


In [10]:
nan_rows = cdf[cdf['year'] == 'nan']
print(nan_rows)

Empty DataFrame
Columns: [country_a, country_b, year, export_mill_usd]
Index: []


In [11]:
cdf = cdf.dropna(subset=['export_mill_usd'])
print(nan_rows)

Empty DataFrame
Columns: [country_a, country_b, year, export_mill_usd]
Index: []


In [12]:
# Remove the last character from all values in a column, run twice
cdf['year'] = cdf['year'].astype(str).str[:-1]

unique = cdf['year'].unique()
print(unique)

['2024.' '2020.' '2021.' '2022.' '2023.' '2008.' '2009.' '2012.' '2013.'
 '2014.' '2017.' '2018.' '1997.' '1998.' '1999.' '1992.' '1993.' '1994.'
 '1995.' '1996.' '2000.' '2002.' '2003.' '2006.' '2016.' '2019.' '2001.'
 '2004.' '2005.' '2007.' '2010.' '2011.' '2015.']


In [13]:
cdf.head(40)

,country_a,country_b,year,export_mill_usd
0,"Uzbekistan, Republic of","Mauritania, Islamic Republic of",2024.,0.070536
1,Holy See,Ukraine,2020.,0.000228
2,Holy See,Ukraine,2021.,0.001300
3,Holy See,Ukraine,2022.,0.000881
4,Holy See,Ukraine,2023.,0.000108
5,Holy See,Ukraine,2024.,0.000001
6,"Uzbekistan, Republic of","Bahamas, The",2023.,0.000002
7,"Uzbekistan, Republic of","Ethiopia, The Federal Democratic Republic of",2023.,0.012100
8,"Eritrea, The State of",Costa Rica,2008.,0.000108
9,"Eritrea, The State of",Costa Rica,2009.,0.000036


In [14]:
cdf.to_csv('../../data/raw/v2clean_imf_trade.csv', index=False)

save it before iso3 mapping

In [15]:
import sys
import pandas as pd

# sys.path.append("C:\\Users\\vanes\\2024 SCHOOL UVA VANESSA YAN\\YEAR 2\\SEM4\\GP\\Violent-Offenders-GPV---CSSci-\\src")
sys.path.append("../../src")

import importlib
import country_utils_extended as cs
importlib.reload(cs)


<module 'country_utils_extended' from 'c:\\Users\\vanes\\2024 SCHOOL UVA VANESSA YAN\\YEAR 2\\SEM4\\GP\\Violent-Offenders-GPV---CSSci-\\notebooks\\raw-data-exploration\\../../src\\country_utils_extended.py'>

In [16]:
cdf = pd.read_csv('../../data/raw/v2clean_imf_trade.csv')

cdf['a_iso3'] = cdf['country_a'].apply(cs.get_iso3)
cdf['b_iso3'] = cdf['country_b'].apply(cs.get_iso3)

In [17]:
cs.check_coverage(cdf, 'a_iso3', 'country_a', 'country a')


=== country a ===
Total rows with country_a: 715086
Matched to ISO3: 715086 (100.0%)
Unmatched unique values: 0


array([], dtype=object)

In [18]:
cs.check_coverage(cdf, 'b_iso3', 'country_b', 'country b')


=== country b ===
Total rows with country_b: 715086
Matched to ISO3: 715086 (100.0%)
Unmatched unique values: 0


array([], dtype=object)

In [19]:
cdf.to_csv('../../data/processed/v2_imf_trade.csv', index=False)

done

In [27]:
country_list = list(cdf['country_a'].value_counts().items())
for country, count in country_list:
    print(f"{country}: {count}")

Germany: 6587
Netherlands, The: 6584
United Kingdom: 6573
France: 6570
Italy: 6564
Denmark: 6552
United States: 6515
Sweden: 6513
Spain: 6498
Austria: 6442
Switzerland: 6436
India: 6435
Japan: 6414
Malaysia: 6406
Finland: 6394
Canada: 6366
Poland, Republic of: 6319
Korea, Republic of: 6316
Australia: 6307
Ireland: 6296
China, People's Republic of: 6275
Brazil: 6257
Norway: 6239
Portugal: 6183
Indonesia: 6155
Greece: 6137
Türkiye, Republic of: 6119
New Zealand: 6109
Hong Kong Special Administrative Region, People's Republic of China: 6019
Pakistan: 6011
Czech Republic: 5782
Thailand: 5750
Hungary: 5637
Bulgaria: 5636
Philippines: 5630
Slovak Republic: 5627
Belgium: 5626
Romania: 5597
Slovenia, Republic of: 5421
Russian Federation: 5365
Cyprus: 5349
Singapore: 5296
Colombia: 5239
United Arab Emirates: 5231
Mexico: 5224
Egypt, Arab Republic of: 5203
Argentina: 5076
Luxembourg: 5073
South Africa: 5052
Ukraine: 5021
Lithuania, Republic of: 5019
Latvia, Republic of: 5017
Lebanon: 4957
Chile:

all done with clean, now merge

In [10]:
# merge trade with team dataset

team_df = pd.read_csv('../../data/merged/panel_dyadic_1992_2024.csv')

merged_df = team_df.merge(
    cdf,
    left_on=['sender_iso3', 'recipient_iso3', 'year'],
    right_on=['a_iso3', 'b_iso3', 'year'],
    how='left'  # keeps all rows in the team dataset
)

# optional: keep only relevant columns
merged_df = merged_df.drop(columns=['a_iso3', 'b_iso3', 'country_a', 'country_b'])

merged_df[['import', 'export']] = merged_df[['import', 'export']].fillna(0)

merged_df = merged_df.rename(columns={
    'import': 'tid_import',
    'export': 'tid_export'
})

In [11]:
merged_df.head()

,sender_iso3,recipient_iso3,year,arms_tiv,bilateral_debt,ucdp_troops,ucdp_weapons,ucdp_training,ucdp_funding,colonial_tie,journalist_killings,oda_received,us_intervention,tid_export,tid_import
0,ABW,ISR,1996,17.80,NaN,0,0,0,0,0,0.0,2.216620e+09,0,0.0,0.0
1,AGO,CIV,2002,1.72,NaN,0,0,0,0,0,0.0,8.361000e+08,1,0.0,0.0
2,ALB,BFA,2011,1.20,NaN,0,0,0,0,0,0.0,5.945600e+08,0,0.0,0.0
3,ARE,BFA,2023,3.97,NaN,0,0,0,0,0,0.0,7.062923e+08,0,0.0,0.0
4,ARE,BGR,2021,0.72,NaN,0,0,0,0,0,0.0,NaN,0,0.0,0.0


Check

In [36]:
# Random sample of 5 rows
merged_df[['sender_iso3', 'recipient_iso3', 'year', 'tid_import', 'tid_export']].sample(5)

,sender_iso3,recipient_iso3,year,tid_import,tid_export
2169,DEU,AUS,2015,0.000000,0.000000
27962,USA,MOZ,2003,0.000000,0.000000
13434,CAN,PRY,2009,8.321097,11.316225
15545,CHN,PRY,2001,0.000000,0.000000
27100,USA,BIH,2003,0.000000,0.000000


In [37]:
cdf.query("a_iso3 == 'CAN' and b_iso3 == 'PRY' and year == 2009")[
    ['import', 'export']
]

import_export,import,export
22545,8.321097,11.316225


In [39]:
merged_df.to_csv('../../data/merged/v2_panel_dyadic_1992_2024.csv', index=False)